In [1]:
from qiskit import transpile

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.transpiler import CouplingMap, generate_preset_pass_manager
from qiskit.visualization import plot_histogram

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeKingston, FakeBoston, FakeTorino, FakePittsburgh, FakeMiami

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.noise import ReadoutError

import numpy as np
from fractions import Fraction
from math import floor, gcd, log, log2
import json
import os
import time

In [26]:
# code from https://quantum.cloud.ibm.com/docs/en/tutorials/shors-algorithm

def a2kmodN (a, k , N):

    for _ in range (k):
        a = int (np.mod (a**2, N))
    return a

In [2]:
# code from https://quantum.cloud.ibm.com/docs/en/tutorials/shors-algorithm
# generalized mod gate that will be used in the future

'''
def modMultGate (b, N):

    if gcd (b, N) > 1:
        print (f"Error: gcd ({b}, {N}) > 1")
    else:
        n = floor (log (N - 1, 2)) + 1
        U = np.full ((2**n , 2**n), 0)
        for x in range (N):
            U[b * x % N][x] = 1
        for x in range (N, 2**n):
            U[x][x] = 1
        G = UnitaryGate (U)
        G.name = f"M_{b}"

        return G
'''

'\ndef modMultGate (b, N):\n\n    if gcd (b, N) > 1:\n        print (f"Error: gcd ({b}, {N}) > 1")\n    else:\n        n = floor (log (N - 1, 2)) + 1\n        U = np.full ((2**n , 2**n), 0)\n        for x in range (N):\n            U[b * x % N][x] = 1\n        for x in range (N, 2**n):\n            U[x][x] = 1\n        G = UnitaryGate (U)\n        G.name = f"M_{b}"\n\n        return G\n'

In [4]:
# rotationGate method based off the description from Yang/Markidis paper designed for a specific family of N
# optimized by AI from an original bubble-swap version to a single cycle-decomposition

def rotationGate (numQubits, shift):

    qc = QuantumCircuit (numQubits)

    shift = shift % numQubits

    if shift != 0:
        visited = [False] * numQubits
        for start in range (numQubits):
            if visited[start]:
                continue
            
            cycle = [start]
            visited[start] = True
            nxt = (start + shift) % numQubits
            
            while nxt != start:
                cycle.append (nxt)
                visited[nxt] = True
                nxt = (nxt + shift) % numQubits

            for i in range (len (cycle) - 1):
                qc.swap (cycle[i], cycle[i + 1])

    G = qc.to_gate()
    G.name = f"Rot{shift}"

    return G

In [5]:
# code from https://quantum.cloud.ibm.com/docs/en/tutorials/shors-algorithm
# generalized mod gate circuit that will be used in the future

'''
N = 31
a = 2
numTarget = floor (log (N - 1, 2)) + 1
numControl = 8

bList = [a2kmodN (a, k, N) for k in range (numControl)]

control = QuantumRegister (numControl, name = "C")
target = QuantumRegister (numTarget, name = "T")
output = ClassicalRegister (numControl, name = "out")
circuit = QuantumCircuit (control, target, output)

circuit.x (target[0])

for qubit, b in zip (control, bList):
    circuit.h (qubit)
    circuit.compose (modMultGate (b, N).control(), qubits = [qubit] + list (target), inplace = True)
        
circuit.compose (QFT (numControl, inverse = True), qubits = control, inplace = True)

circuit.measure (control, output)
'''

'\nN = 31\na = 2\nnumTarget = floor (log (N - 1, 2)) + 1\nnumControl = 8\n\nbList = [a2kmodN (a, k, N) for k in range (numControl)]\n\ncontrol = QuantumRegister (numControl, name = "C")\ntarget = QuantumRegister (numTarget, name = "T")\noutput = ClassicalRegister (numControl, name = "out")\ncircuit = QuantumCircuit (control, target, output)\n\ncircuit.x (target[0])\n\nfor qubit, b in zip (control, bList):\n    circuit.h (qubit)\n    circuit.compose (modMultGate (b, N).control(), qubits = [qubit] + list (target), inplace = True)\n\ncircuit.compose (QFT (numControl, inverse = True), qubits = control, inplace = True)\n\ncircuit.measure (control, output)\n'

In [6]:
# code modified from https://quantum.cloud.ibm.com/docs/en/tutorials/shors-algorithm

def makeCircuit (N, a, t, approxDegree = 0):

    numTarget = floor (log (N - 1, 2)) + 1
    
    bList = [a2kmodN (a, k, N) for k in range (t)]
    
    control = QuantumRegister (t, name = "C")
    target = QuantumRegister (numTarget, name = "T")
    output = ClassicalRegister (t, name = "out")
    circuit = QuantumCircuit (control, target, output)
    
    circuit.x (target[0])
    
    for qubit, b in zip (control, bList):
        circuit.h (qubit)
        assert b > 0 and (b & (b - 1)) == 0
        shift = int (log2 (b))
        circuit.compose (rotationGate (numTarget, shift).control(), qubits = [qubit] + list (target), inplace = True)
            
    circuit.compose (QFT (t, inverse = True, approximation_degree = approxDegree), qubits = control, inplace = True)
    
    circuit.measure (control, output)

    return circuit

In [7]:
# code from https://quantum.cloud.ibm.com/docs/en/tutorials/shors-algorithm
# Generalized mod gate circuit that will be used in the future

'''
couplingMap = CouplingMap.from_line (14)
pm = generate_preset_pass_manager (coupling_map = couplingMap)
transpiledCirc = pm.run (circuit)
deep = transpiledCirc
for _ in range (10):
    deep = deep.decompose()

print (f"2q-depth:  {deep.depth (lambda x: x.operation.num_qubits == 2)}")
print (f"2q-size:  {deep.size (lambda x: x.operation.num_qubits == 2)}")
print (f"Operator counts:  {deep.count_ops()}")
#deep.decompose().draw (output = "mpl", fold = -1, style = "clifford", idle_wires = False)
'''

'\ncouplingMap = CouplingMap.from_line (14)\npm = generate_preset_pass_manager (coupling_map = couplingMap)\ntranspiledCirc = pm.run (circuit)\ndeep = transpiledCirc\nfor _ in range (10):\n    deep = deep.decompose()\n\nprint (f"2q-depth:  {deep.depth (lambda x: x.operation.num_qubits == 2)}")\nprint (f"2q-size:  {deep.size (lambda x: x.operation.num_qubits == 2)}")\nprint (f"Operator counts:  {deep.count_ops()}")\n#deep.decompose().draw (output = "mpl", fold = -1, style = "clifford", idle_wires = False)\n'

In [8]:
# code from https://quantum.cloud.ibm.com/docs/en/tutorials/shors-algorithm
# seed candidate optimization from AI, used to cut down time from running on backend due to time constraints

def runExperiment (backend, circuit, shots = 1024):

    start = time.perf_counter()
    best = None
    for seed in range (8):
        candidate = transpile (circuit, backend, optimization_level = 1, seed_transpiler = seed)
        n2q = candidate.count_ops().get('cz', 0) + candidate.count_ops().get('cx', 0)
        if best is None or n2q < best[0]:
            best = (n2q, candidate)
    compiled = best[1]
    #compiled = transpile (circuit, backend, optimization_level = 1)
    #print (compiled.count_ops())
    #print ("Transpile:", time.perf_counter() - start)

    start = time.perf_counter()
    job = backend.run (compiled, shots = shots)
    #print ("Submitted:", time.perf_counter() - start)

    start = time.perf_counter()
    results = job.result()
    counts = results.get_counts()
    #print (results.results[0].metadata.get ('method'))
    #print ("Result:", time.perf_counter() - start)

    return counts

In [9]:
# noise code from https://deepwiki.com/Qiskit/qiskit-aer/7-noise-simulation

def createNoiseModel (error1, error2, readoutError):

    error1 = depolarizing_error (error1, 1)
    error2 = depolarizing_error (error2, 2)
    
    readoutError = ReadoutError ([[1 - readoutError, readoutError], [readoutError, 1 - readoutError]])

    noiseList = [("Ideal", None)]

    noiseModel1 = NoiseModel()
    noiseModel1.add_all_qubit_quantum_error (error1, ["h", "x"])
    noiseList.append (("1Q", noiseModel1))
    
    noiseModel2 = NoiseModel()
    noiseModel2.add_all_qubit_quantum_error (error2, ["cx"])
    noiseList.append (("2Q", noiseModel2))
    
    noiseModel3 = NoiseModel()
    noiseModel3.add_all_qubit_readout_error (readoutError)
    noiseList.append (("Readout", noiseModel3))
    
    noiseModel4 = NoiseModel()
    noiseModel4.add_all_qubit_quantum_error (error1, ["h", "x"])
    noiseModel4.add_all_qubit_quantum_error (error2, ["cx"])
    noiseModel4.add_all_qubit_readout_error (readoutError)
    noiseList.append (("Combined", noiseModel4))

    return noiseList

In [11]:
# backends defined at https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/fake-provider
# selected backends to mimic Yang/Markidis paper

backendList = []
backendList.append (FakeKingston())
backendList.append (FakeBoston())
backendList.append (FakeTorino())
backendList.append (FakePittsburgh())
#backendList.append (FakeMiami())

In [12]:
# autocorrelation peak strength (Apeak) is defined in the first row of Table 2 of the Yang/Markidis paper

def calculateApeak (count, Q):

    numBits = int (log2 (Q))
    shots = sum (count.values())
    py = [count.get (format (y, f'0{numBits}b'), 0) / shots for y in range (Q)]
    uy = 1 / Q
    qy = [p - uy for p in py]
    
    A = []
    for l in range (Q):
        Al = sum (qy[y] * qy[(y + l) % Q] for y in range (Q))
        A.append (Al)

    if A[0] == 0:
        return 0
        
    return max (A[1:]) / A[0]

In [13]:
# normalized entropy (Hnorm) is defined in the second row of Table 2 of the Yang/Markidis paper

def calculateHnorm (count, Q):

    shots = sum (count.values())
    py = [c / shots for c in count.values()]
    top = -sum (p * log (p) for p in py if p > 0)
    Hnorm = top / log (Q)
    return Hnorm

In [14]:
# r is defined in section 2 of the Yang/Markidis paper

def calculateR (y, Q, N, a):

    if y == 0:
        return None

    r0 = Fraction (y, Q).limit_denominator (N).denominator

    if r0 <= 1:
        return None

    if pow (a, r0, N) == 1:
        return r0

    return None

In [28]:
# m is defined in section 2 of the Yang/Markidis paper

def calculateM (count, Q, N, a):

    shots = sum (count.values())
    m = {}

    for bitstring, c in count.items():

        y = int (bitstring, 2)
        p = c / shots
        r = calculateR (y, Q, N, a)

        if r is None:
            continue

        m[r] = m.get (r, 0) + p
        
    return m

In [16]:
# dominant verified mass fraction (M1frac) is defined in the third row of Table 2 of the Yang/Markidis paper

def calculateM1frac (m):

    Mver = sum (m.values())
    
    if Mver == 0:
        return 0

    sortedM = sorted (m.values(), reverse = True)
    M1 = sortedM[0]

    return M1 / Mver

In [17]:
# verified margin fraction (DeltaVerFrac) is defined in the fourth row of Table 2 of the Yang/Markidis paper

def calculateDeltaVerFrac (m):

    Mver = sum (m.values())
    
    if Mver == 0:
        return 0

    sortedM = sorted (m.values(), reverse = True)
    M1 = sortedM[0]
    M2 = sortedM[1] if len (sortedM) > 1 else 0

    return (M1 - M2) / Mver

In [18]:
# Function to avoid having to calculate m twice

def calculatePostProcessingAware (count, Q, N, a):

    m = calculateM (count, Q, N, a)

    return calculateM1frac (m), calculateDeltaVerFrac (m)

In [19]:
# true order as defined in Section 2 of the Yang/Markidis paper

def calculateTrueOrder (a, N):

    r = 1

    while pow (a, r, N) != 1:
        r += 1

    return r

In [20]:
# the definition of recoverable is defined in section 2 of the Yang/Markidis paper

def calculateRecoverable (count, Q, N, a, trueOrder):

    m = calculateM (count, Q, N, a)

    if not m:
        return False

    predictedOrder = max (m, key = m.get)

    return predictedOrder == trueOrder

In [42]:
# function needs to be optimized and cleaned up
# left as is due to time constraints

def runOnSimulator (simulatorList, isSimulator, isShort):
    
    dataset = []

    if isShort:
        #NList = [3, 7, 15]
        NList = [3]
    else:
        NList = [3, 7, 15, 31, 63, 127]
    
    for N in NList:

        aList = [2, 4, 8, 16]
        #aList = [2]
        
        for a in aList:
    
            if gcd (a, N) != 1:
                continue
    
            trueOrder = calculateTrueOrder (a, N)
    
            if trueOrder <= 1:
                continue

            tList = [8, 10]
            #tList = [10]
            
            for t in tList:

                approxList = [0, 1, 2]
                #approxList = [2]

                for approx in approxList:
        
                    Q = 2**t
                    circuit = makeCircuit (N, a, t, approx)
    
                    if isSimulator:
    
                        for name, noise in simulatorList:

                            print (N, a, t, approx, name)
    
                            simulator = AerSimulator (noise_model = noise)
                            counts = runExperiment (simulator, circuit, 4000)
    
                            Apeak = calculateApeak (counts, Q)
                            Hnorm = calculateHnorm (counts, Q)
                            M1frac, DeltaVerFrac = calculatePostProcessingAware (counts, Q, N, a)
                            recoverable = calculateRecoverable (counts, Q, N, a, trueOrder)
            
                            dataset.append ({
                                "N":  N,
                                "a":  a,
                                "t":  t,
                                "Q":  Q,
                                "approximationDegree":  approx,
                                "noise":  name,
                                "counts":  counts,
                                "Apeak":  Apeak,
                                "Hnorm":  Hnorm,
                                "M1frac":  M1frac,
                                "DeltaVerFrac":  DeltaVerFrac,
                                "trueOrder":  trueOrder,
                                "recoverable":  recoverable
                            })
    
                    else:
    
                        for backend in simulatorList:

                            print (N, a, t, approx, backend.name)

                            simulator = AerSimulator.from_backend (backend)
                            counts = runExperiment (simulator, circuit, 4000)
                            
                            Apeak = calculateApeak (counts, Q)
                            Hnorm = calculateHnorm (counts, Q)
                            M1frac, DeltaVerFrac = calculatePostProcessingAware (counts, Q, N, a)
                            recoverable = calculateRecoverable (counts, Q, N, a, trueOrder)
            
                            dataset.append ({
                                "N":  N,
                                "a":  a,
                                "t":  t,
                                "Q":  Q,
                                "approximationDegree":  approx,
                                "backend":  backend.name,
                                "counts":  counts,
                                "Apeak":  Apeak,
                                "Hnorm":  Hnorm,
                                "M1frac":  M1frac,
                                "DeltaVerFrac":  DeltaVerFrac,
                                "trueOrder":  trueOrder,
                                "recoverable":  recoverable
                            })
                        
    return dataset

In [37]:
def saveDataset (filename, dataset):

    if not dataset:
        return

    with open (filename, "w") as f:
        json.dump (dataset, f, indent = 4)

In [31]:
noiseList = createNoiseModel (0.08, 0.16, 0.15)

simulatorDataset = runOnSimulator (noiseList, True, True)

C:\Users\Rampidzier\AppData\Local\Temp\ipykernel_43772\3678536435.py:22: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit.compose (QFT (t, inverse = True, approximation_degree = approxDegree), qubits = control, inplace = True)


3 2 8 0 Ideal
3 2 8 0 1Q
3 2 8 0 2Q
3 2 8 0 Readout
3 2 8 0 Combined
3 2 8 1 Ideal
3 2 8 1 1Q
3 2 8 1 2Q
3 2 8 1 Readout
3 2 8 1 Combined
3 2 8 2 Ideal
3 2 8 2 1Q
3 2 8 2 2Q
3 2 8 2 Readout
3 2 8 2 Combined
3 2 10 0 Ideal
3 2 10 0 1Q
3 2 10 0 2Q
3 2 10 0 Readout
3 2 10 0 Combined
3 2 10 1 Ideal
3 2 10 1 1Q
3 2 10 1 2Q
3 2 10 1 Readout
3 2 10 1 Combined
3 2 10 2 Ideal
3 2 10 2 1Q
3 2 10 2 2Q
3 2 10 2 Readout
3 2 10 2 Combined
3 8 8 0 Ideal
3 8 8 0 1Q
3 8 8 0 2Q
3 8 8 0 Readout
3 8 8 0 Combined
3 8 8 1 Ideal
3 8 8 1 1Q
3 8 8 1 2Q
3 8 8 1 Readout
3 8 8 1 Combined
3 8 8 2 Ideal
3 8 8 2 1Q
3 8 8 2 2Q
3 8 8 2 Readout
3 8 8 2 Combined
3 8 10 0 Ideal
3 8 10 0 1Q
3 8 10 0 2Q
3 8 10 0 Readout
3 8 10 0 Combined
3 8 10 1 Ideal
3 8 10 1 1Q
3 8 10 1 2Q
3 8 10 1 Readout
3 8 10 1 Combined
3 8 10 2 Ideal
3 8 10 2 1Q
3 8 10 2 2Q
3 8 10 2 Readout
3 8 10 2 Combined


In [40]:
numRecoverable = sum (exp["recoverable"] for exp in simulatorDataset)
numFailed = len (simulatorDataset) - numRecoverable

filename = "simulatorDataset.json"

print (f"Recoverable: {numRecoverable}")
print (f"Not recoverable: {numFailed}")

saveDataset (filename, simulatorDataset)

Recoverable: 60
Not recoverable: 0


In [43]:
backendDataset = runOnSimulator (backendList, False, True)

C:\Users\Rampidzier\AppData\Local\Temp\ipykernel_43772\3678536435.py:22: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit.compose (QFT (t, inverse = True, approximation_degree = approxDegree), qubits = control, inplace = True)


3 2 8 0 fake_kingston
3 2 8 0 fake_boston
3 2 8 0 fake_torino
3 2 8 0 fake_pittsburgh


C:\Users\Rampidzier\AppData\Local\Temp\ipykernel_43772\3678536435.py:22: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit.compose (QFT (t, inverse = True, approximation_degree = approxDegree), qubits = control, inplace = True)


3 2 8 1 fake_kingston
3 2 8 1 fake_boston
3 2 8 1 fake_torino
3 2 8 1 fake_pittsburgh


C:\Users\Rampidzier\AppData\Local\Temp\ipykernel_43772\3678536435.py:22: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit.compose (QFT (t, inverse = True, approximation_degree = approxDegree), qubits = control, inplace = True)


...

3 8 10 0 fake_kingston
3 8 10 0 fake_boston
3 8 10 0 fake_torino
3 8 10 0 fake_pittsburgh


C:\Users\Rampidzier\AppData\Local\Temp\ipykernel_43772\3678536435.py:22: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit.compose (QFT (t, inverse = True, approximation_degree = approxDegree), qubits = control, inplace = True)


3 8 10 1 fake_kingston
3 8 10 1 fake_boston
3 8 10 1 fake_torino
3 8 10 1 fake_pittsburgh


C:\Users\Rampidzier\AppData\Local\Temp\ipykernel_43772\3678536435.py:22: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit.compose (QFT (t, inverse = True, approximation_degree = approxDegree), qubits = control, inplace = True)


3 8 10 2 fake_kingston
3 8 10 2 fake_boston
3 8 10 2 fake_torino
3 8 10 2 fake_pittsburgh


In [47]:
numRecoverable = sum (exp["recoverable"] for exp in backendDataset)
numFailed = len (backendDataset) - numRecoverable

N = backendDataset[0]["N"]
filename = f"backendDatasetN{N}.json"

print (f"Recoverable: {numRecoverable}")
print (f"Not recoverable: {numFailed}")

saveDataset (filename, backendDataset)

Recoverable: 48
Not recoverable: 0
